# Monitoreo de Data Drift con Evidently (referencia de entrenamiento vs periodo actual)

Este notebook compara con **Evidently** los datos con los que se entrenó el modelo de **P2** (riesgo de déficit, balance o excedente en las próximas 3 horas) contra un periodo posterior que el modelo no vio al entrenar.

- **Referencia:** `split = train`, del 2024-01-15 al 2025-12-31 (el 15 de enero es el primer día con el lag de 336 h completo).
- **Actual:** `split = test`, de jul a sep 2026, el periodo más reciente de los datos abiertos de MiBici.

### Objetivos
1. Preparar ambos conjuntos con `prepare_monitoring_data()` y unir las predicciones del champion.
2. Generar el reporte de **data drift** por variable y de **calidad de datos** (`DataDriftPreset` + `DataSummaryPreset`).
3. Revisar la evolución por mes y separar la estacionalidad comparando contra los mismos meses de la referencia.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from mi_bici.monitoring.evidently_drift import (
    DEFAULT_REPORT_PATH,
    DRIFT_CAT_METHOD,
    DRIFT_NUM_METHOD,
    DRIFT_THRESHOLD,
    STATIC_COLUMNS,
    additional_comparisons,
    compare,
    load_monitoring_frames,
    sample_rows,
)

print(f'Pruebas: {DRIFT_NUM_METHOD} (numéricas), {DRIFT_CAT_METHOD} (categóricas), umbral {DRIFT_THRESHOLD}')

## 1. Conjuntos de referencia y actual
Se cargan completos y después se toma una muestra de 5000 filas de cada uno para el reporte.

In [ ]:
reference, current = load_monitoring_frames()

for name, df in [('Referencia', reference), ('Actual', current)]:
    print(f'{name}: {df.shape[0]:,} filas, {df["hour_ts"].min():%Y-%m-%d} a {df["hour_ts"].max():%Y-%m-%d}')

ref_sample = sample_rows(reference, 5000)
curr_sample = sample_rows(current, 5000)

## 2. Reporte de Evidently: drift por variable y calidad de datos
Wasserstein (normalizada) y Jensen-Shannon son **distancias**, no p-values: hay drift cuando el score es **mayor o igual** al umbral de 0.1.

In [ ]:
main_comparison = compare(ref_sample, curr_sample, output_html_path=DEFAULT_REPORT_PATH)
print(f'Reporte HTML: {DEFAULT_REPORT_PATH}')

drift_table = pd.DataFrame(main_comparison['columns']).T
drift_table['fija'] = drift_table.index.isin(STATIC_COLUMNS)
drift_table.sort_values('score', ascending=False)

In [ ]:
print(f"Variables con drift: {main_comparison['drifted_columns']}")
print(f"Proporción sin variables fijas: {main_comparison['drifted_share_monitored']:.0%}")
pd.Series(main_comparison['data_quality_current'], name='actual')

## 3. Evolución por mes y comparación contra los mismos meses
- `train_vs_<mes>`: referencia completa contra cada mes del periodo actual.
- `mismos_meses_train_vs_actual`: solo jul-sep de 2024 y 2025 contra jul-sep 2026, para separar la estacionalidad de un cambio real.

In [ ]:
comparisons = {'train_vs_actual': main_comparison, **additional_comparisons(reference, current)}

scores = pd.DataFrame(
    {name: {col: d['score'] for col, d in c['columns'].items()} for name, c in comparisons.items()}
).drop(index=STATIC_COLUMNS)
scores.round(3).style.background_gradient(axis=None, vmin=0, vmax=0.2)

## 4. Distribuciones de las variables con más cambio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for df, label in [(ref_sample, 'Referencia (train)'), (curr_sample, 'Actual (jul-sep 2026)')]:
    axes[0].hist(df['pressure_last_3h'].clip(-1, 1), bins=40, alpha=0.5, density=True, label=label)
axes[0].set_title('pressure_last_3h (recortada a [-1, 1])')
axes[0].legend()

monthly = current.groupby(current['hour_ts'].dt.to_period('M'))[['departures', 'arrivals']].mean()
monthly.loc['Referencia'] = reference[['departures', 'arrivals']].mean()
monthly.plot.bar(ax=axes[1], title='Salidas y llegadas promedio por hora')

classes = pd.DataFrame({
    'Referencia': reference['risk_class'].value_counts(normalize=True),
    'Actual': current['risk_class'].value_counts(normalize=True),
})
classes.plot.bar(ax=axes[2], title='Proporción de risk_class')
plt.tight_layout()

## 5. Lectura de los resultados

Los scores salen de la muestra de 5000 filas por conjunto (semilla 42); las medias, desviaciones y proporciones, de los conjuntos completos.

1. **Variables de presión con drift.** `pressure_last_3h` y los lags `pressure_window_lag_24h/168h/336h` superan el umbral (Wasserstein ≈ 0.11-0.13). En el periodo actual su dispersión es menor (desviación estándar ≈ 0.33 vs 0.37): hay menos horas con presión extrema.
2. **No es solo estacionalidad.** Contra jul-sep de 2024 y 2025, `pressure_last_3h` y los lags de 168 h y 336 h siguen por encima del umbral, aunque con distancias menores.
3. **El drift cambia por mes.** Julio es el mes más alejado de la referencia (también `net_flow`). En septiembre las presiones bajan del umbral, pero `departures` sí deriva: el promedio por hora sube de ≈ 3.6 en jul-ago a ≈ 4.5, coherente con el regreso a clases.
4. **Objetivo y predicciones sin drift.** `risk_class` (0.04) y la predicción del champion (0.04) quedan bajo el umbral, aunque la proporción de déficit baja de 12.1 % a 9.2 % y la de excedente de 12.8 % a 10.0 %.
5. **Calidad de datos.** El periodo actual no tiene valores faltantes, filas o columnas vacías ni columnas constantes.

Los scores cercanos al umbral pueden cambiar con otra muestra. La interpretación detallada y las implicaciones para el desempeño del modelo se documentan en `docs/monitoreo.md`.